In [1]:
import numpy
%matplotlib inline
import matplotlib.pyplot as plt
def identify_bad_nodes_old(file, metrics=["metrics_0", "metrics_1", "metrics_2", "metrics_3", "metrics_4", "metrics_5"], threshold=0.):
    for i, metric in enumerate(metrics):
        print("\tMetric:", metric)
        h5pointer = file["node"][metric]
        num_cycles = h5pointer.shape[0]
        # eliminate first 3rd of run
        start = num_cycles - 2*num_cycles//3
        start = 0
        print("\t\t", h5pointer.shape, start)
        data = h5pointer[start:]
        print("\t\t", data.shape, data.min(), data.max())
        bad = numpy.argwhere(data<=threshold)
        if bad.shape[0] == 0:
            print("\t\tNo bad nodes! Yeah for success")
        else:
            bad_nodes = sorted(set(bad[:,1]))
            print("\t\tFor",metric, "There are", len(bad_nodes), "actual bad nodes", bad.shape, bad[0], bad[-1])
            print("\t\tTotal bad_nodes was", bad.shape[0], "which is ", bad.shape[0]/data.size*100., "%")
            mn = 12222222
            mx = -1.E34
            truly_bad =[]
            constant = []
            for b in bad_nodes:
                indices = numpy.argwhere(bad[:,1]==b)
                mn = min(mn, indices.shape[0])
                mx = max(mx, indices.shape[0])
                node = h5pointer[:,b]
                diff = numpy.abs(node[:-1] - node[1:])
                #print(b, diff.max())
                if diff.max() !=0:
                    line, = plt.plot(node)
                    line.set_label(str(b))
                    truly_bad.append(b)
                else:
                    constant.append(b)
                    #all_bad = True
                    #for i in range(6,32):
                    #    some_metric = file["node"]["metrics_{}".format(i)][:,b]
                    #    diff = numpy.abs(some_metric[:-1] - some_metric[1:])
                    #    if diff.max() != 0:
                    #        all_bad = False
                    #        #print("Metrics", i, "varies at node" ,b)
                    #        break
                    #if all_bad:
                    #    print("Node", b, "has all metrics constant")
                    
            #    out.append([b, numpy.take(bad[:,0], indices).min()])
            #    print(numpy.take(bad[:,0], indices))
            #print("mn, mx", mn, mx)
            plt.legend()
            plt.show()
            plt.savefig(metric)
            plt.clf()
            plt.cla()
    return truly_bad, constant


def identify_bad_nodes(filename,
                       metrics=["metrics_0", "metrics_1", "metrics_2", "metrics_3", "metrics_4", "metrics_5"],
                       jsonfile="bad_nodes.json",
                       jsonkey=None,
                       threshold=0., verbose=False):
    out = {}
    if os.path.exists(jsonfile):
        with open(jsonfile) as f:
            existing = json.load(f)
    else:
        existing = {}
    if jsonkey is None:
        jsonkey = filename
    if jsonkey in existing:
        return existing[jsonkey]
    for i, metric in enumerate(metrics):
        if verbose:
            print("\tMetric:", metric)
        h5pointer = h5py.File(filename, mode="r")["node"][metric]
        num_cycles = h5pointer.shape[0]
        # eliminate first 3rd of run
        # start = num_cycles - 2*num_cycles//3
        # or look at all cycles
        start = 0
        if verbose:
            print("\t\t", h5pointer.shape, start)
        data = h5pointer[start:]
        if verbose:
            print("\t\t", data.shape, data.min(), data.max())
        bad = numpy.argwhere(data<=threshold)
        out[metric] = []
        if bad.shape[0] == 0:
            print("\t\tNo bad nodes! Yeah for success")
        else:
            bad_nodes = list(set(bad[:,1]))
            if verbose:
                print("\t\tFor",metric, "There are", len(bad_nodes), "actual bad nodes", bad.shape, bad[0], bad[-1])
                print("\t\tTotal bad_nodes was", bad.shape[0], "which is ", bad.shape[0]/data.size*100., "%")
            for b in bad_nodes:
                indices = numpy.argwhere(bad[:,1]==b)
                node = h5pointer[:,b]
                if node.max() != 0:
                    out[metric].append([int(b), h5pointer.shape[0], numpy.take(bad[:,0], indices)[:,0].astype("i").tolist()])
    if not use_dask:
        print("Dumping:", out)
        existing[jsonkey] = out
        with open(jsonfile,"w") as f:
            json.dump(existing, f)
    return out

In [2]:
error = '/p/lscratchh/cdoutrix/cdoutrix/IBM/workaround/base/molar.1_shock.1.7_taper.0.6_skew.0.8/slurm-340974.out'
no_error = '/p/lscratchh/cdoutrix/cdoutrix/IBM/workaround/base/molar.0.0_shock.1.1_taper.0.6_skew.0.8/slurm-340673.out'
error_goal_reached = '/p/lscratchh/cdoutrix/cdoutrix/IBM/workaround/base/molar.0.7_shock.1.3_taper.0.9_skew.0.9/slurm-340863.out'
custom = '/p/lscratchh/cdoutrix/cdoutrix/IBM/workaround/base/molar.0.3_shock.1.5_taper.0.4_skew.0.8/slurm-340789.out'

In [3]:
import h5py, os
ds = error_goal_reached
truly_bad = []
constant = []
#for ds in [error, error_goal_reached, no_error]:
for ds in [custom,]:
    run = ds.split("/")[-2]
    name = os.path.join(os.path.dirname(ds), "IBM", "res.2_"+run+".hdf5")
    print("**************************************************************************************")
    print("**************************************************************************************")
    print("name:", name)
    print("**************************************************************************************")
    print("**************************************************************************************")
    #h5 = h5py.File(name)
    bad = identify_bad_nodes(name, jsonfile="bad_me.json")
    #constant.append(const)


**************************************************************************************
**************************************************************************************
name: /p/lscratchh/cdoutrix/cdoutrix/IBM/workaround/base/molar.0.3_shock.1.5_taper.0.4_skew.0.8/IBM/res.2_molar.0.3_shock.1.5_taper.0.4_skew.0.8.hdf5
**************************************************************************************
**************************************************************************************


KeyboardInterrupt: 

In [ ]:
!more bad_me.json

In [ ]:
[ len(const) for const in constant]

In [ ]:
numpy.allclose(sorted(constant[-1][]), sorted(constant[0])

In [ ]:
for i in range(401):
    if constant[0][i] != constant[1][i]:
        print("bad at:", i)
        break

In [ ]:
c0 = constant[0]
c1 = constant[1]

In [ ]:
c0[i], c1[i]

In [ ]:
c1.pop(c1.index(2716))

In [ ]:
for i in range(401):
    if c0[i] != c1[i]:
        print("bad at:", i)
        break